In [ ]:
import os
from pathlib import Path
from datetime import datetime
import json
import shutil

# INPUTS ----------------------------------------------------------------------------------------------------

# file path -------------------------------------------------------------------------------------------------
data_dir = ''
raw_dir  = ''
decoder_dir = ''
# save decoder

# sample path
sample_dir = ''
milestone = 0

# Decoder setting --------------------------------------------------------------------------------------------
device_num = 0
sub_num = 0
selected_chan = ''

# 'condition', 'condition + generated', 'condition + actual' 
# 'condition + artificial'
# 'generated' 

In [ ]:
# make decoder dir -------------------------------------------------------------------------------------------
Path(decoder_dir).mkdir(exist_ok=True)

# Trainer ----------------------------------------------------------------------------------------------------
y_dim = '3d' # '3d'
k = 5 # for decoder

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = str(device_num)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import pytorch_model_summary
import gc

import numpy as np
import mne
import os

import math
from scipy.integrate import cumtrapz

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import sys
import matplotlib

if sys.platform.startswith('win'):
    # Windows
    # matplotlib.use('Qt5Agg')
    print("Using Qt5Agg backend")
    %matplotlib qt
else:
    # Linux / headless
    matplotlib.use('Agg')
    print("Using Agg backend")
    
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# load data

subs = {
    1:  {'name': '',    
         'bads': [], # bad channel name
         'depths': [], # non-ecog channel name
         'session': 0,  # session to use
         'drop_s1': [], # bad trial num
         'drop_s2': [], # bad trial num
         'notch': [], # notch filter (not used)
         'n_session1': 0}, # number of trials in session 1
}

sub_name = subs[sub_num]['name']
session_num = subs[sub_num]['session']

# 1. dataset

In [ ]:
def find_max_distance_timepoint(y, trig=100):
    # y shape: (3, seq_len, #trial)
    # trig   : timepoints of task trigger 
    start_points = y[:, trig, :]  
    subsequences = y[:, trig:, :]
    distances = np.linalg.norm(subsequences - start_points[:, np.newaxis, :], axis=0)
    max_distance_indices = np.argmax(distances, axis=0)
    reach_times = max_distance_indices + trig
    return reach_times


# load data (2 session) and make event epochs (X, y)
def load_data(sub_num, X_mode, y_mode, dsfreq=100):
    sub_name = subs[sub_num]['name']
    # set parameters -------------------------------------------------------------------------------------------------------------------------------------------
    rmv = np.array([], dtype=int)  # save bad trial idx
    accumulated_trial = 0 # count total trial
    
    # nsteps: for sequence dataset
    tmin, tmax = 1, 3
    
    # initialize (self.X, self.y가 아님)
    # X data
    X_ME = np.zeros([0, 0, 0])
    X_MI = np.zeros([0, 0, 0])
    # y data
    y_vel = np.zeros([3, 0, 0])
    y_pos =  np.zeros([3, 0, 0])
    # sequence (4 direction label)
    sequences = np.zeros([0,])
        
    # 1. load all session data -----------------------------------------------------------------------------------------------------------------------------------
    for session in range(1, session_num + 1):
        
        # load data ---
        fname = f""
        epochs = mne.read_epochs(fname, preload=True)
        
        epochs_ME = epochs['ME']
        epochs_MI = epochs['MI']
        
        # load meta data ---
        fname = f"{}.npz"
        meta = np.load(fname)
        
        pos = meta['pos'] # (3, datapoint, trials)
        vel = meta['vel'] # (3, datapoint, trials)
        sequence = meta['sequence'] # (trial, 1)

        # load channel info ---
        fname = f'{}.pkl'
        with open(fname, 'rb') as f:
            ch_info = pickle.load(f)
    
        # get parameters
        sfreq = epochs.info["sfreq"]
        s_ratio = int(sfreq/dsfreq) 
    
        picks = mne.pick_types(epochs.info, meg=True, eeg=True, eog=True, stim=True, exclude='bads')
        nchan = len(picks)
    
        data_len = len(range(-np.int64(np.round(sfreq*tmin)), np.int64(np.round(sfreq*tmax)) + 1, s_ratio)) 
        # == TO REMOVE BAD TRIALS ==
        drop = f'drop_s{session}'
        drops =  subs[sub_num][drop]
    
        me_rmv = np.array([], dtype=int); mi_rmv = np.array([], dtype=int)
        for idx in drops: 
            n = idx // 2
            if idx % 2 == 0: me_rmv = np.append(me_rmv, n)    # ME              
            else: mi_rmv = np.append(mi_rmv, n)                #MI
    
        if X_mode == 'ME': rmv = np.append(rmv, me_rmv + accumulated_trial)
        else: rmv = np.append(rmv, mi_rmv + accumulated_trial)

        # save sequence
        sequences = np.append(sequences, sequence)
    
        # Make X, y data -----------------------------------------------------------------------------------------------------------------------------
    
        # initialize
        if X_ME.shape[-1] == 0:
            X_ME = np.zeros([nchan, data_len, 0]); X_MI = np.zeros([nchan, data_len, 0])
            y_vel = np.zeros([3, data_len, 0]);    y_pos =  np.zeros([3, data_len, 0])
           
        # Downsampling: make X data (epochs, chan, datapoints) --> (chan, datapoitns, epochs) 
        data_ME = epochs_ME.get_data(copy=True) # (epochs, channels, time points)
        seg_ME = data_ME[:, :, ::s_ratio]
        seg_ME = np.transpose(seg_ME, (1, 2, 0)) # (chan, datapoints, epochs)
        X_ME = np.append(X_ME, seg_ME, axis = 2)
    
        data_MI = epochs_MI.get_data(copy=True)
        seg_MI = data_MI[:, :, ::s_ratio]
        seg_MI = np.transpose(seg_MI, (1, 2, 0))
        X_MI = np.append(X_MI, seg_MI, axis = 2)
        
        # Downsampling: make y data
        seg_vel = vel[:, ::s_ratio, :]
        y_vel = np.append(y_vel, seg_vel, axis =2)
    
        seg_pos = pos[:, ::s_ratio, :]
        y_pos = np.append(y_pos, seg_pos, axis =2)
    
        print('\n >> %d Events' %(X_ME.shape[-1])) # session 1 / session 2 Events
        accumulated_trial += X_ME.shape[-1]
    
                   
    # 2. set X, y ----------------------------------------------------------------------------------------------------------------------------------
    
    # Set data (remove bad trial idx for y data)
    if X_mode == 'ME': X = X_ME
    elif X_mode == 'MI': X = X_MI
    
    if y_mode == 'pos': y = y_pos; label = y_pos
    elif y_mode == 'vel': y = y_vel; label = y_pos

    # find time for reaching task
    reach_times = find_max_distance_timepoint(label, trig=100)
        
    X = np.delete(X, rmv, axis=2)
    y = np.delete(y, rmv, axis=2)
    label = np.delete(label, rmv, axis=2)
    sequences = np.delete(sequences, rmv)       # 4 direction label
    reach_times = np.delete(reach_times, rmv)

    print(f'X shape          : {X.shape}')            # (chan, datapoints, #epochs)
    print(f'y shape          : {y.shape}')            # (chan, datapoints, #epochs)
    print(f'label shape      : {label.shape}')        
    print(f'sequence shape   : {sequences.shape}')     # (#epochs, )
    print(f'reach_times shape: {reach_times.shape}')
    
    return X, y, label, sequences, reach_times, epochs, ch_info

# --------------------------------------------------------------------------
def make_even_seq(data):  
    even_data=[]
    if data.shape[1] % 2 == 1: 
        even_data = data[:, :-1, :]
    return even_data

In [ ]:
class ECoGDataset(torch.utils.data.Dataset):

    def __init__(self, sub_num, tmin, tmax, X, y, label, dsfreq=100, nsteps=10):

        # set timepoints
        if np.isscalar(tmax):
            tmax = np.full(y.shape[2], np.round(dsfreq * (tmax + 1)), dtype=np.int64)
        
        t_start = np.int64(np.round(dsfreq * (tmin+1)))
        
        # initialize
        xx = np.zeros((0, nsteps, X.shape[0]))  # (n, 10, # chan)     10 points = 0.1 sec (100 Hz)
        yy = np.zeros((0, y.shape[0]))          # (n, 3)   
        ll = np.zeros((0, label.shape[0]))
        self.seq_len = np.zeros(label.shape[2], dtype=np.int64)

        # (chan, datapoints, epochs) --> (epochs, datapoints, chan)
        for n, data in enumerate(np.transpose(X, [2,1,0])): # n: epoch num / data : (datapoints, chan)
            
            # save seq_len for each trial
            t_end = tmax[n]
            interval = np.int64(np.round(t_end - t_start))
            self.seq_len[n] = interval
            
            # xx
            trial = np.zeros((interval, 0, data.shape[1]))
            
            for i in range(nsteps): 
                steps = data[t_start-nsteps+i : t_start-nsteps+i+interval, :].reshape([interval, 1, data.shape[1]])
                trial = np.append(trial, steps, axis=1)
                
            xx = np.append(xx, trial, axis=0)
            
            # yy (chan, datapoints, epochs) --> (epochs, datapoints, chan)
            yy = np.append(yy, np.transpose(y[ :, t_start-nsteps+i : t_start-nsteps+i+interval, n], [1,0]), axis=0) # i = 9
            ll = np.append(ll, np.transpose(label[:, t_start-nsteps+i : t_start-nsteps+i+interval, n], [1,0]), axis=0)
            
            # count progress
            print('.',end='')     
            if (n+1)%100 == 0: print(n+1)
        
        # 3. Return self.X, self.y ; make tensor and normalization ----------------------------------------------------------------------------------------------------

        # numpy to tensor
        self.X=torch.from_numpy(xx).float() 
        self.y=torch.from_numpy(yy).float()
        self.label = torch.from_numpy(ll).float()

        # Normalization
        self.means = {}
        self.stds = {}
        
        # X
        self.means['X'] = [self.X[:,0,i].mean() for i in range(self.X.shape[-1])]
        self.stds['X'] = [self.X[:,0,i].std() for i in range(self.X.shape[-1])]
        
        for i in range(self.X.shape[-1]):
            self.X[:,:,i] = (self.X[:,:,i] - self.means['X'][i]) / self.stds['X'][i] / 3

        # y, label
        self.means['y'] = [self.y[:,i].mean() for i in range(self.y.shape[-1])]
        self.stds['y'] = [self.y[:,i].std() for i in range(self.y.shape[-1])]
        self.means['label'] = [self.label[:,i].mean() for i in range(self.label.shape[-1])]
        self.stds['label'] = [self.label[:,i].std() for i in range(self.label.shape[-1])]

        for i in range(self.y.shape[-1]):
            self.y[:,i] = (self.y[:,i] - self.means['y'][i]) / self.stds['y'][i] / 3
            self.label[:,i] = (self.label[:,i] - self.means['label'][i]) / self.stds['label'][i] / 3
        
        print(f'\n >> X shape: {self.X.shape}, y shape: {self.y.shape}') # Xsize=(sets,n_step, # chan), Ysize=(sets,3)

        # trial idx
        self.start_idx = np.cumsum(self.seq_len)
        self.start_idx = np.insert(self.start_idx[:-1], 0, 0)

    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.label[idx], self.means, self.stds, self.seq_len, self.start_idx


In [ ]:
def get_chan_idx(channels, total_ch_names):
    indices = []
    for ch in channels:
        if str(ch) in total_ch_names:
            indices.append(total_ch_names.index(str(ch)))
        else:
            print(f"Warning: Chan {ch} not found in total_ch_names.")
    return indices

In [ ]:
import json
import numpy as np
import pickle

# load data ------------------------------------------------------------------------
X_mode = 'ME'
y_mode = 'vel' 
dsfreq = 100  

X, y, label, sequence, reach_times, epochs, ch_info = load_data(sub_num, X_mode, y_mode, dsfreq) # (chan, datapoints, #trial)

X = make_even_seq(X)
y = make_even_seq(y)
label = make_even_seq(label)

# for inter case
artificial_data = []
train_sub = []

for sample_path in sample_dir:
    # model info ------------------------
    mdl_info_path = Path(sample_path) / 'used_mdl_info.json'
    
    with open(mdl_info_path, 'r') as f:
        mdl_info = json.load(f)
    
    train_session = mdl_info['train_session']
    rand_cond_flag = mdl_info['rand_cond_flag']
    n_target_chan = len(mdl_info['target'])
    train_sub_num = mdl_info["sub_num"]
    train_sub.append(train_sub_num)
    train_sub_name = subs[train_sub_num]['name']
    
    if rand_cond_flag:
        print('---- RANDOM CONDTION ----')

    # sample info ------------------------
    
    sample_info_path = Path(sample_path) / 'sample_info.json'
    
    with open(sample_info_path, 'r') as file:
        sample_info = json.load(file)
    
    
    target_chan = sample_info['target']
    condition_chan = sample_info['condition']

    # load generated sample  ----------
    loaded_data = np.load(f"{}.npz")
       
    # ground-truth EXISTS
    # INTRA
    if target_chan:
        for i, chan in enumerate(target_chan):
            globals()[f'generated_[{chan}]'] = loaded_data['X_generated'][i:i+1,:,:]
            print(f'generated_[{chan}]: {globals()[f"generated_[{chan}]"].shape}')
    
    # INTER
    else:
        X_artificial = loaded_data['X_generated']
        artificial_data.append(X_artificial)

# INTER
if artificial_data:
    artificial_data = np.concatenate(artificial_data, axis=0)

# remove duplicated & sort
target_chan = sorted(list(set(target_chan)))
condition_chan = sorted(list(set(condition_chan)))

# samle path 
print(f'target chan: {target_chan}')
print(f'condition chan: {condition_chan} (# cond: {len(condition_chan)})')

# y_2d/3d ----------------------------------------------------------------
if y_dim == '2d':
    y = y[1:3, :, :]
    label = label[1:3,:,:]
elif y_dim == '3d':
    pass
else:
    print('error')

# set mdl dir

if len(sample_dir) == 1:
    # intra ---
    if target_chan:
        sub_dir = Path(f'{decoder_dir}/sub{}')
    # inter ---
    else:
        mdl_target = mdl_info['target']
        sub_dir = Path(f'{decoder_dir}/sub{sub_num}')

else:
    sub_dir = Path(f'{decoder_dir}/sub{sub_num')


sub_dir.mkdir(parents=True ,exist_ok=True)

mdl_dir = sub_dir.joinpath(f'{}')
mdl_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# chan name, chan loc (for XAI)
chan_name = []
chan_loc = []


# INTRA (ground-truth EXISTS)
# sample info ------------------------   
sample_info_path = Path(sample_dir[0]) / 'sample_info.json'
with open(sample_info_path, 'r') as file:
    sample_info = json.load(file)

sample_sub_num = sample_info["sub_num"]
sample_sub_name = subs[sample_sub_num]['name']

fname = f"{raw_dir}/{sample_sub_name}_session_1_raw.fif"
sample_raw = mne.io.read_raw_fif(fname, preload=True)

chan_name = sample_raw.info['ch_names']
chan_loc = np.array([ch['loc'][:3] for ch in sample_raw.info['chs']])

# INTER (ADD artificial chan)
if not target_chan:    
    for sample_path in sample_dir:
        # model info ------------------------
        mdl_info_path = Path(sample_path) / 'used_mdl_info.json'
        
        with open(mdl_info_path, 'r') as f:
            mdl_info = json.load(f)
        
        train_session = mdl_info['train_session']
        train_sub_num = mdl_info["sub_num"]
        train_target = mdl_info["target"]
        train_sub_name = subs[train_sub_num]['name']
        
        fname = f"{raw_dir}/{train_sub_name}_session_1_raw.fif"
        train_raw = mne.io.read_raw_fif(fname, preload=True)

        total_chan_name = train_raw.info['ch_names']
        total_chan_loc = np.array([ch['loc'][:3] for ch in train_raw.info['chs']])
        art_idx = get_chan_idx(train_target, total_chan_name)

        for i in range(len(train_target)):
            chan_name.append(f'{train_target[i]}(s{train_sub_num})')
            chan_loc = np.vstack((chan_loc, total_chan_loc[art_idx[i]]))

print(len(chan_name))
print(chan_loc.shape)

In [ ]:
print('\nMaking X to decode---------------------')

# test session ---------------------------------------------------------

n_session1 = subs[sub_num]['n_session1']

if train_session == 0:
    pass
    
elif train_session == 1:
    # test: session2
    X = X[:,:,n_session1:]
    y = y[:,:,n_session1:]
    label = label[:,:,n_session1:]
    sequence = sequence[n_session1:]
    reach_times = reach_times[n_session1:]

elif train_session == 2:
    # test: session1
    X = X[:,:,:n_session1]
    y = y[:,:,:n_session1]
    label = label[:,:,:n_session1]
    sequence = sequence[:n_session1]
    reach_times = reach_times[:n_session1]

else:
    raise ValueError(f"Invalid train_session value: {train_session}. Must be either 0, 1, or 2.")
    

print(f'X shape                 : {X.shape}')            # (chan, datapoints, #epochs)
print(f'y shape                 : {y.shape}')            # (chan, datapoints, #epochs)
print(f'label shape             : {label.shape}')        
print(f'sequence shape          : {sequence.shape}')     # (#epochs, )

# find idx from ch_names --------------------------------------------------------
total_ch_names = epochs.info.ch_names

target_idx = get_chan_idx(target_chan, total_ch_names)
cond_idx = get_chan_idx(condition_chan, total_ch_names)

# CAR ---------------------------------------------------------------------------

X_CAR = X - np.mean(X, axis=0, keepdims=True)  # X: (#chan, timepoints, trial)

# g-t exists
if target_chan:
    # generated
    X_gen_replaced = np.array(X, copy=True)
    
    for i, idx in enumerate(target_idx):
        chan = target_chan[i]
        X_gen_replaced[idx] = globals()[f'generated_[{chan}]']
        print(f'chan {chan} replaced')
    
    X_gen_CAR = X_gen_replaced - np.mean(X_gen_replaced, axis=0, keepdims=True)

# g-t NOT exist
else:
    print(f'artificial data shape: {artificial_data.shape}')
    X_art_concat = np.concatenate((X.copy(), artificial_data), axis=0)
    X_art_CAR = X_art_concat - np.mean(X_art_concat, axis=0, keepdims=True) # (#chan, timepoints, trial)

# chan to decode ----------------------------------------------------------------
X_to_decode = []

if selected_chan == 'all':
    X_to_decode = np.copy(X_CAR)

elif selected_chan == 'actual':
    X_to_decode = np.copy(X_CAR)[target_idx, :, :]

elif selected_chan == 'condition':
    X_to_decode = np.copy(X_CAR)[cond_idx, :, :]

elif selected_chan == 'generated':
    X_to_decode = np.copy(X_gen_CAR)[target_idx, :, :]

elif selected_chan == 'drop_target':
    X_to_decode = np.delete(X_CAR, target_idx, axis=0)

elif selected_chan =='condition + generated':
    cond_gen_idx = sorted(np.concatenate((target_idx, cond_idx)))
    X_to_decode = X_gen_CAR[cond_gen_idx, :, :]

elif selected_chan == 'condition + actual':
    cond_gen_idx = sorted(np.concatenate((target_idx, cond_idx)))
    X_to_decode = np.copy(X_CAR)[cond_gen_idx, :, :]

elif selected_chan == 'condition + artificial':
    X_cond = np.copy(X_art_CAR)[cond_idx, :, :] 
    X_art = np.copy(X_art_CAR)[-n_target_chan:, :, :]
    X_to_decode = np.concatenate((X_cond, X_art), axis=0)

elif selected_chan == 'all + artificial':
    X_to_decode = np.copy(X_art_CAR)
    
elif selected_chan == 'drop_target + generated':
    X_to_decode = X_gen_CAR

else:
    print(f'!! Selected_chan is chan {selected_chan} !!')
    sel_idx = sorted([x - 1 for x in selected_chan])
    X_to_decode = X[sel_idx, :, :]

print(f'X_to_decode             : {selected_chan}')
print(f'X_to_decode shape       : {X_to_decode.shape}')


In [ ]:
# make dataset for lstm ------------------------------------------------------------
tmin = -0.5   
tmax = reach_times
nsteps = 10
n_batch = 512

ecog_dataset=ECoGDataset(sub_num, tmin, tmax, X_to_decode, y, label, dsfreq, nsteps)

# 2. model

## basic blstm

In [ ]:
torch.manual_seed(1)

class move_decoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_outputs, drop_out):
        
        super(move_decoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm1 = nn.LSTM(input_size, hidden_size[0], num_layers, bidirectional=True, batch_first=True)
        self.dropout1 = nn.Dropout(drop_out)
        self.lstm2 = nn.LSTM(hidden_size[0]*2, hidden_size[1], num_layers, bidirectional=True, batch_first=True) # batch_first=True
        self.dropout2 = nn.Dropout(drop_out)
        self.fc1 = nn.Linear(hidden_size[1]*2, hidden_size[2])
        self.dropout3 = nn.Dropout(drop_out)
        self.fc2 = nn.Linear(hidden_size[2], num_outputs)

    def forward(self, x): 
        h0 = torch.zeros(2*self.num_layers, x.size(0), self.hidden_size[0]).to(device)
        c0 = torch.zeros(2*self.num_layers, x.size(0), self.hidden_size[0]).to(device)
        h1 = torch.zeros(2*self.num_layers, x.size(0), self.hidden_size[1]).to(device)
        c1 = torch.zeros(2*self.num_layers, x.size(0), self.hidden_size[1]).to(device) 

        self.lstm1.flatten_parameters()
        out, _ = self.lstm1(x) 
        out    = F.tanh(out)
        out    = self.dropout1(out)
        self.lstm2.flatten_parameters()
        out, _ = self.lstm2(out)
        out    = F.tanh(out)
        out    = self.dropout2(out)
        out    = self.fc1(out[:,-1,:])
        out    = F.relu(out)
        out    = self.dropout3(out)
        out    = self.fc2(out)
        
        return out

## 2-1. training

In [ ]:
from sklearn.model_selection import KFold

# 5 - fold cross validation
kf = KFold(n_splits=k, shuffle=False)

n_epochs = X.shape[-1]
trial_indices = np.arange(n_epochs)

for fold, (train_val_idx, test_idx) in enumerate(kf.split(trial_indices)):
    print(f'Fold {fold} | Train + Val ({train_val_idx.size}) / Test ({test_idx.size}): {test_idx[0]}~{test_idx[-1]}')

In [ ]:
from torchinfo import summary

# set model dimension
input_dim = X_to_decode.shape[0] # num channel
hidden_dim = [32, 16, 64]
num_layer = 1
output_dim = y.shape[0] # 3 dir
drop_out = 0.2

# set model, criterion, optimizer
model = move_decoder(input_dim, hidden_dim, num_layer, output_dim, drop_out).to(device)

# print model summary
print(summary(model,  torch.zeros(n_batch, 10, input_dim).shape))

# set model parameters
learning_rate = 0.0005
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
num_epochs = 100


# Time measurement
starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True) # timer
total_timing = 0.0

In [ ]:
# functions

# set directory
loss_plot_dir = mdl_dir.joinpath('loss_plots')
loss_plot_dir.mkdir(parents=True, exist_ok=True)

def initialize_model(input_dim, hidden_dim, num_layer, output_dim, drop_out, learning_rate):
    model = move_decoder(input_dim, hidden_dim, num_layer, output_dim, drop_out).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    return model, optimizer

def plot_loss(all_train_loss, all_val_loss, sub_num, sub_name, loss_plot_dir):
    
    for i, (train_loss, val_loss) in enumerate(zip(all_train_loss, all_val_loss), 1):
        plt.figure(figsize=(10, 5))
        plt.plot(train_loss, label=f'Fold {i} Train Loss')
        plt.plot(val_loss, label=f'Fold {i} Val Loss')
        
        plt.title(f'Loss Plot for Sub {sub_num} {sub_name} - Fold {i}')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        
        # save for each fold
        plt.show()
        plt.savefig(os.path.join(loss_plot_dir, f'{sub_num}_{sub_name}_fold_{i}_loss_plot.png'))
        plt.close()

def make_subset(dataset, idx):
    
    start_idx = dataset.start_idx
    seq_len = dataset.seq_len
    
    indices = []
    for i in idx:
        indices.extend(range(start_idx[i], start_idx[i] + seq_len[i]))
    subset = torch.utils.data.Subset(dataset, indices)
    
    return subset

### train loop

In [ ]:
# train loop

# to save loss from all fold 
all_train_loss = []
all_val_loss = []

# CV loop ---------------------------------------------------------------------------------------------
for fold, (train_val_idx, test_idx) in enumerate(kf.split(trial_indices)):
    
    print(f"Starting fold {fold+1}")
    model, optimizer = initialize_model(input_dim, hidden_dim, num_layer, output_dim, drop_out, learning_rate)

    num_train = int(train_val_idx.size*0.8)
    train_idx = train_val_idx[:num_train]
    val_idx = train_val_idx[num_train:]
    
    train_subset = make_subset(ecog_dataset, train_idx)
    val_subset = make_subset(ecog_dataset, val_idx)

    train_loader = torch.utils.data.DataLoader(dataset=train_subset, batch_size=n_batch, shuffle=False)
    val_loader = torch.utils.data.DataLoader(dataset=val_subset, batch_size=n_batch, shuffle=False)

    # to save fold loss
    fold_train_loss = []
    fold_val_loss = []

    # Early Stopping parameter
    best_val_loss = float('inf')
    patience_counter = 0
    patience = 10

    for epoch in range(num_epochs):
        
        model.train()
        starter.record() # timer
        total_loss = 0.
        
        # train loop -------------------------------------------------------
        for x_batch, y_batch, *_ in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            # forward
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)

            # backpropagation & optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        total_loss /= len(train_loader)
        fold_train_loss.append(total_loss)    

        # validation loop -------------------------------------------------
        model.eval()
        val_loss = 0.
        with torch.no_grad():
            for x_val, y_val, *_ in val_loader:
                x_val = x_val.to(device)
                y_val = y_val.to(device)
                outputs = model(x_val)
                val_loss += criterion(outputs, y_val).item()

        val_loss /= len(val_loader)
        fold_val_loss.append(val_loss)

       
        # print epoch results -----------------------------------------------       
        ender.record() # timer
        torch.cuda.synchronize() # waits for everything to finish running
        
        print(f'Fold {fold+1}, Epoch [{epoch+1}/{num_epochs}] Train Loss: {total_loss:.4f} Val Loss: {val_loss:.4f} Time : {starter.elapsed_time(ender)/1000:.4}')
        timing = starter.elapsed_time(ender) 
        total_timing += timing # save total time

        # Early Stopping -----------------------------------------------------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # save model
            torch.save(model.state_dict(), mdl_dir/f'fold_{fold+1}.pt')
            print('mdl saved')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early Stopping t_startgered")
                break

    print('Fold Total time = {:.3f} sec'.format(total_timing/1000))

    all_train_loss.append(fold_train_loss)
    all_val_loss.append(fold_val_loss)


# loss plot & save
plot_loss(all_train_loss, all_val_loss, sub_num, sub_name, loss_plot_dir)

# 3. Evalulation

## eval (3d)

In [ ]:
def calculate_pcc(preds, targets, seq_length):
    num_samples = preds.shape[0] 
    correlations = {'x': [], 'y': [], 'z': []}

    start = 0
    for length in seq_length:
        end = start + length
        
        if end > num_samples:
            print('idx error')
            break

        for i, axis in enumerate(['x', 'y', 'z']):
            segment_preds = preds[start:end, i]
            segment_target = targets[start:end, i]
            
            if len(segment_preds) > 1:
                corr_matrix = torch.corrcoef(torch.stack((segment_preds, segment_target)))
                if corr_matrix.numel() > 1: 
                    correlations[axis].append(corr_matrix[0, 1].item())

        start = end

    cc_x = sum(correlations['x']) / len(correlations['x']) if correlations['x'] else None
    cc_y = sum(correlations['y']) / len(correlations['y']) if correlations['y'] else None
    cc_z = sum(correlations['z']) / len(correlations['z']) if correlations['z'] else None

    valid_ccs = [cc for cc in (cc_x, cc_y, cc_z) if cc is not None]
    mean_cc = sum(valid_ccs) / len(valid_ccs) if valid_ccs else None

    return mean_cc, cc_x, cc_y, cc_z, correlations


def calculate_rmse(preds, targets, seq_length):
    num_samples = preds.shape[0]
    rmses = {'x': [], 'y': [], 'z': []}
    all_3d_rmse = []

    start = 0
    for length in seq_length:
        end = start + length
        if end > num_samples:
            print('idx error')
            break

        for i, axis in enumerate(['x', 'y', 'z']):
            segment_preds = preds[start:end, i]
            segment_target = targets[start:end, i]

            if len(segment_preds) > 1:              
                # Calculate RMSE
                rmse = torch.sqrt(torch.mean((segment_preds - segment_target) ** 2))
                rmses[axis].append(rmse.item())

        # Calculate the 3D RMSE for the current trial
        rmse_3d = math.sqrt(rmses['x'][-1]**2 + rmses['y'][-1]**2 + rmses['z'][-1]**2)
        all_3d_rmse.append(rmse_3d)

        start = end

    # Calculate mean RMSE
    rmse_x = sum(rmses['x']) / len(rmses['x']) if rmses['x'] else None
    rmse_y = sum(rmses['y']) / len(rmses['y']) if rmses['y'] else None
    rmse_z = sum(rmses['z']) / len(rmses['z']) if rmses['z'] else None
    mean_3d_rmse = sum(all_3d_rmse) / len(all_3d_rmse) if all_3d_rmse else None

    return mean_3d_rmse, rmse_x, rmse_y, rmse_z


def integrate_preds(preds, labels, seq_length):
    # preds: [total_samples, 3]
    num_samples = preds.size(0)
    dt = 1 / dsfreq
    integrated_preds = torch.zeros_like(preds)

    start = 0
    for length in seq_length:
        if start >= num_samples:
            break
        
        current_pos = labels[start].unsqueeze(0).clone()  # starting from real position data # current_pos = torch.zeros(1, 3, device=preds.device)  # starting from zero
        integrated_preds[start] = current_pos
        for j in range(1, length):
            current_pos += preds[start + j - 1] * dt
            integrated_preds[start + j] = current_pos

        start += length

    return integrated_preds


def denormalize(tensor, means, stds):
    denormalized = torch.zeros_like(tensor)
    for i in range(tensor.shape[1]):  # denormalizing y (pos/vel data)
        denormalized[:, i] = (tensor[:, i] * stds[i] * 3) + means[i]
    return denormalized

def normalize(tensor, means, stds):
    normalized =  torch.zeros_like(tensor)
    for i in range(tensor.shape[1]):
        normalized[:, i] = (tensor[:, i] - means[i]) / stds[i] # / 3
    return normalized
    
# mdl load & evaluate
def evaluate(model_path, data_loader, device, input_dim, hidden_dim, layer_dim, drop_out, output_dim):

    # load model
    model = move_decoder(input_dim, hidden_dim, layer_dim, output_dim, drop_out).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    preds, targets, labels = [], [], []

    with torch.no_grad():
        for X, y, label, _, _, _, _ in data_loader:
            X, y, label = X.to(device), y.to(device), label.to(device)
            pred = model(X)
            preds.append(pred)
            targets.append(y)
            labels.append(label)
          
    # Concatenate all batches
    preds = torch.cat(preds, dim=0)
    targets = torch.cat(targets, dim=0)
    labels = torch.cat(labels, dim=0)
    
    return preds, targets, labels


In [ ]:
import sys

result_fname = mdl_dir/f"decoding_results.txt"

def prints(*args, **kwargs):
    with open(result_fname, "a") as f:
        print(*args, **kwargs)
        print(*args, **kwargs, file=f)


# evluation for all fold - cc & predicted vs actual plot
fold_results = {
    'train': {'vel': {'cc': [], 'rmse': []}, 'pos': {'cc': [], 'rmse': []}}, # mean, x, y, z
    'test': {'vel': {'cc': [], 'rmse': []}, 'pos': {'cc': [], 'rmse': []}}
}

all_test_preds = []; all_test_preds_pos = []
all_test_targets = []; all_test_labels = []
all_test_sequences = []

for fold, (train_val_idx, test_idx) in enumerate(kf.split(trial_indices)):

    # load model and dataloader
    model_path = mdl_dir/f'fold_{fold+1}.pt'
    num_train = int(train_val_idx.size*0.8)
    train_idx = train_val_idx[:num_train]
    train_seq = sequence[train_idx]
    train_seq_len = ecog_dataset.seq_len[train_idx]

    train_subset = make_subset(ecog_dataset, train_idx)
    train_loader = torch.utils.data.DataLoader(dataset=train_subset, batch_size=n_batch, shuffle=False)
    
    test_seq = sequence[test_idx]
    test_seq_len = ecog_dataset.seq_len[test_idx]
    
    test_subset = make_subset(ecog_dataset, test_idx)
    test_loader = torch.utils.data.DataLoader(dataset=test_subset, batch_size=n_batch, shuffle=False)
    prints(f'== Fold {fold+1} ==  train: {train_seq.size}, test: {test_seq.size}')

    vel_mean = ecog_dataset.means['y'];     vel_std = ecog_dataset.stds['y']
    pos_mean = ecog_dataset.means['label']; pos_std = ecog_dataset.stds['label']
    total_seq_len = ecog_dataset.seq_len

    # eval
    # trainsets -------------------------------------------------------------------------------------------------------------------------------------------------
    train_preds, train_targets, train_labels = evaluate(model_path, train_loader, device, input_dim, hidden_dim, num_layer, drop_out, output_dim)
    # denorm --> integrate --> norm
    # denormalization
    train_preds_denorm = denormalize(train_preds, vel_mean, vel_std); train_targets_denorm = denormalize(train_targets, vel_mean, vel_std)
    train_labels_denorm = denormalize(train_labels, pos_mean, pos_std)
    # integrate
    train_preds_pos_denorm = integrate_preds(train_preds_denorm, train_labels_denorm, train_seq_len)
    # normalization
    train_targets = train_targets * 3; train_labels = train_labels * 3;
    train_preds_pos = normalize(train_preds_pos_denorm, pos_mean, pos_std)
    train_preds = train_preds * 3
    
   
    # cc & rmse
    # 1. vel
    train_vel_cc = calculate_pcc(train_preds, train_targets, train_seq_len)
    train_vel_rmse = calculate_rmse(train_preds, train_targets, train_seq_len)
    fold_results['train']['vel']['cc'].append(train_vel_cc[0:-1])
    fold_results['train']['vel']['rmse'].append(train_vel_rmse)
    prints(f'fold {fold+1} | Train Velocity    | CC: {train_vel_cc[0]:.3f} (X: {train_vel_cc[1]:.3f}, Y: {train_vel_cc[2]:.3f}, Z: {train_vel_cc[3]:.3f}) || RMSE: {train_vel_rmse[0]:.3f} (X: {train_vel_rmse[1]:.3f}, Y: {train_vel_rmse[2]:.3f}, Z: {train_vel_rmse[3]:.3f})')
    # 2. pos
    train_pos_cc = calculate_pcc(train_preds_pos, train_labels, train_seq_len)
    train_pos_rmse = calculate_rmse(train_preds_pos, train_labels, train_seq_len)
    fold_results['train']['pos']['cc'].append(train_pos_cc[0:-1])
    fold_results['train']['pos']['rmse'].append(train_pos_rmse)
    prints(f'fold {fold+1} | Train Pos (recon) | CC: {train_pos_cc[0]:.3f} (X: {train_pos_cc[1]:.3f}, Y: {train_pos_cc[2]:.3f}, Z: {train_pos_cc[3]:.3f}) || RMSE: {train_pos_rmse[0]:.3f} (X: {train_pos_rmse[1]:.3f}, Y: {train_pos_rmse[2]:.3f}, Z: {train_pos_rmse[3]:.3f})')

     
    # testsets ----------------------------------------------------------------------------------------------------------------------------------------------------
    test_preds, test_targets, test_labels, *_ = evaluate(model_path, test_loader, device, input_dim, hidden_dim, num_layer, drop_out, output_dim)
    # denorm --> integrate --> norm
    # denormalization
    test_preds_denorm = denormalize(test_preds, vel_mean, vel_std); test_targets_denorm = denormalize(test_targets, vel_mean, vel_std)
    test_labels_denorm = denormalize(test_labels, pos_mean, pos_std)
    
    # integrate
    test_preds_pos_denorm = integrate_preds(test_preds_denorm, test_labels_denorm, test_seq_len)

    # normalization
    test_targets = test_targets * 3; test_labels = test_labels * 3
    test_preds_pos = normalize(test_preds_pos_denorm, pos_mean, pos_std)
    test_preds = test_preds * 3

     # 1. vel cc
    test_vel_cc = calculate_pcc(test_preds, test_targets, test_seq_len)
    test_vel_rmse = calculate_rmse(test_preds, test_targets, test_seq_len)
    fold_results['test']['vel']['cc'].append(test_vel_cc[0:-1])
    fold_results['test']['vel']['rmse'].append(test_vel_rmse)
    prints(f'fold {fold+1} | Test Velocity     | CC: {test_vel_cc[0]:.3f} (X: {test_vel_cc[1]:.3f}, Y: {test_vel_cc[2]:.3f}, Z: {test_vel_cc[3]:.3f}) || RMSE: {test_vel_rmse[0]:.3f} (X: {test_vel_rmse[1]:.3f}, Y: {test_vel_rmse[2]:.3f}, Z: {test_vel_rmse[3]:.3f})')
    # 2. pos cc
    test_pos_cc = calculate_pcc(test_preds_pos, test_labels, test_seq_len)
    test_pos_rmse = calculate_rmse(test_preds_pos, test_labels, test_seq_len)
    fold_results['test']['pos']['cc'].append(test_pos_cc[0:-1])
    fold_results['test']['pos']['rmse'].append(test_pos_rmse)
    prints(f'fold {fold+1} | Test Pos (recon)  | CC: {test_pos_cc[0]:.3f} (X: {test_pos_cc[1]:.3f}, Y: {test_pos_cc[2]:.3f}, Z: {test_pos_cc[3]:.3f}) || RMSE: {test_pos_rmse[0]:.3f} (X: {test_pos_rmse[1]:.3f}, Y: {test_pos_rmse[2]:.3f}, Z: {test_pos_rmse[3]:.3f})')

    # save all fold
    all_test_preds.append(test_preds.cpu())   
    all_test_preds_pos.append(test_preds_pos.cpu())
    all_test_targets.append(test_targets.cpu())
    all_test_labels.append(test_labels.cpu())
    
    concatenated_test_preds = np.concatenate(all_test_preds, axis=0)
    concatenated_test_preds_pos = np.concatenate(all_test_preds_pos, axis=0) 
    concatenated_test_targets = np.concatenate(all_test_targets, axis=0)
    concatenated_test_labels = np.concatenate(all_test_labels, axis=0)


prints("== Average Results Across All Folds ==")

# Test ------------------------------------
# Test - vel
avg_cc = np.mean(fold_results['test']['vel']['cc'], axis=0)
avg_rmse = np.mean(fold_results['test']['vel']['rmse'], axis=0)
prints(f'Average | Test Velocity | CC: {avg_cc[0]:.5f} (X: {avg_cc[1]:.5f}, Y: {avg_cc[2]:.5f}, Z: {avg_cc[3]:.5f}) || RMSE: {avg_rmse[0]:.5f} (X: {avg_rmse[1]:.5f}, Y: {avg_rmse[2]:.5f}, Z: {avg_rmse[3]:.5f})')


# Test - pos
avg_cc = np.mean(fold_results['test']['pos']['cc'], axis=0)
avg_rmse = np.mean(fold_results['test']['pos']['rmse'], axis=0)
prints(f'Average | Test Pos (recon) | CC: {avg_cc[0]:.5f} (X: {avg_cc[1]:.5f}, Y: {avg_cc[2]:.5f}, Z: {avg_cc[3]:.5f}) || RMSE: {avg_rmse[0]:.5f} (X: {avg_rmse[1]:.5f}, Y: {avg_rmse[2]:.5f}, Z: {avg_rmse[3]:.5f})\n')

In [ ]:
## SAVE DECODED RESULTS

np.save(os.path.join(mdl_dir, "vel_gt.npy"),  concatenated_test_targets)
np.save(os.path.join(mdl_dir, "vel_pred.npy"), concatenated_test_preds)
np.save(os.path.join(mdl_dir, "pos_gt.npy"),  concatenated_test_labels)
np.save(os.path.join(mdl_dir, "pos_pred.npy"), concatenated_test_preds_pos)
np.save(os.path.join(mdl_dir, "sequence.npy"), sequence)
np.save(os.path.join(mdl_dir, "total_seq_len.npy"), total_seq_len)

# 4. XAI

In [ ]:
# directory for saving results
save_dir = Path(mdl_dir).joinpath('XAI')
save_dir.mkdir(parents=True, exist_ok=True)

result_fname = save_dir/f"IGS.txt"

## 4-1. Save Node (inter)

In [ ]:
# save .node for BrainNet

# directory for saving results
node_dir = Path(save_dir).joinpath('BrainNet')
node_dir.mkdir(parents=True, exist_ok=True)

def artificial_BrainNet(chan_name, chan_loc, output_fname="output.node"):
    with open(output_fname, 'w') as f:
        for i, ch_name in enumerate(chan_name):
            locs = chan_loc[i]
            dot_size = 1
            chan_type = 0.9
            if i > (X.shape[0] - 1):
                chan_type = 0.1
            f.write(f"{locs[0]}\t{locs[1]}\t{locs[2]}\t{chan_type}\t{dot_size}\t{ch_name}\n")

# artificial chan mark
if not target_chan:
    artificial_BrainNet(chan_name, chan_loc, node_dir/f"[Art_Mark]_{}.node")


## get igs

In [ ]:
import copy
from tqdm import tqdm

def get_gradients(model, inputs):

    model.train(True)
    
    for out_n in model.parameters():
        model_outsize=out_n.shape
        
    model_outsize=model_outsize[0] # 3
    device=next(model.parameters()).device
    grads=torch.zeros((len(inputs), inputs[0].shape[0], inputs[0].shape[1], inputs[0].shape[2], model_outsize), device=device)
    
    for interp_num, interp_input in enumerate(inputs):
        interp_input = interp_input.to(device)
        interp_input.requires_grad_(True)
        interp_input.retain_grad()
        y=model(interp_input) # y.shape: torch.Size([1(b), 3])

        for i in range(y.shape[-1]):
            y_part=y.gather(1, torch.ones([inputs[0].shape[0],1],dtype=torch.int64,device=device)*i)
            interp_input.grad = None
            grad_tensor = torch.ones_like(y_part) 
            y_part.backward(grad_tensor, retain_graph=True)
            grads[interp_num,:,:,:,i]=interp_input.grad # shape: [step_num, b, seq_len, channels, 3]
    
    return grads

def get_integrated_gradients(model, x, baseline=None, num_steps=50):
    """
    igs = get_integrated_gradients(model, x, baseline=None, num_steps=100)
    model    : pytorch DNN model
    x        : reference input
    baseline : default (None) indicates 0
    num_steps: number of steps from baseline to the reference input
    """
    device = next(model.parameters()).device
    
    if baseline is None:
        baseline = torch.zeros_like(x, device=device)
    else:
        baseline = baseline.float().to(device)

    # 1. Interpolation to reconstruct step inputs
    x=x.float().to(device)
    inputs=[]

    for step in range(1,num_steps+1):
        interpolated_signal1=(baseline+(step/num_steps)*(x-baseline)).clone().detach()
        interpolated_signal1.requires_grad_(True)
        inputs.append(interpolated_signal1)

    # 2. Remove random effects from the model
    n_model = copy.deepcopy(model)
    for child in n_model.children():
        if isinstance(child, nn.Dropout): # drop rate = 0
            child.p = 0

    # 3. Get gradients
    grads=get_gradients(n_model, inputs) # shape: [step_num, b, seq_len, channels, 3]
    # Approximate the integral using the trapezoidal rule
    grads = (grads[:-1] + grads[1:]) / 2.0

    # 4. Calculate integrated gradients   
    igs = (x - baseline).unsqueeze(3).repeat(1,1,1,grads.shape[-1]) * grads.mean(0)  # shape: [b, 10, 204, 3]

    return igs

In [ ]:
import sys

# get igs from all fold models
test_igs = []
train_igs = []

for fold, (train_val_idx, test_idx) in enumerate(kf.split(trial_indices)):

    # load model and dataloader
    model_path = f'fold_{fold+1}.pt'
    model = move_decoder(input_dim, hidden_dim, num_layer, output_dim, drop_out)
    model.load_state_dict(torch.load(model_path))
    model = model.to(device)
    
    num_train = int(train_val_idx.size*0.8)
    train_idx = train_val_idx[:num_train]
    
    train_subset = make_subset(ecog_dataset, train_idx)
    train_loader = torch.utils.data.DataLoader(dataset=train_subset, batch_size=n_batch, shuffle=False)
    
    test_subset = make_subset(ecog_dataset, test_idx)
    test_loader = torch.utils.data.DataLoader(dataset=test_subset, batch_size=n_batch, shuffle=False)
    print(f'\n == Fold {fold+1} ==  train: {train_idx.size}, test: {test_idx.size}')

    
    # trainsets -------------------------------------------------------------------------------------------------------------------------------------------------
    for X, _, _, _, _, _, _ in tqdm(train_loader, total=len(train_loader), desc='Collecting Trainset IGS'):
    
        igs = get_integrated_gradients(model, X, num_steps=100) # shape: [b, time (10), chan, 3]
        igs = torch.mean(igs.abs(), dim=1)[:, :, 1:].mean(2) #[b, chan]

        train_igs.append(igs)
    
    ## testsets ----------------------------------------------------------------------------------------------------------------------------------------------------  
    for X, _, _, _, _, _, _ in tqdm(test_loader, total=len(test_loader), desc='Collecting Testset IGS'):
        igs = get_integrated_gradients(model, X, num_steps=100) 
        igs = torch.mean(igs.abs(), dim=1)[:, :, 1:].mean(2) #[b, chan]
        test_igs.append(igs)

all_train_igs = torch.cat(train_igs, dim=0)
all_test_igs = torch.cat(test_igs, dim=0)

mean_train_igs = all_train_igs.mean(dim=0)
mean_test_igs = all_test_igs.mean(dim=0)

print('IGs size: ', mean_train_igs.shape)
print('IGs size: ', mean_test_igs.shape)


# save
np.savez(save_dir/f'igs.npz', mean_train_igs=mean_train_igs.cpu().numpy())